# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library. We'll systematically load, preview, and visually analyze the data, referencing all dataset components by their Croissant schema `@id` fields.

### Dataset Source
Croissant schema URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and inspect the core description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL (as provided)
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
dataset = mlc.Dataset(croissant_url)

# Show dataset name and description
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Let's list all available record sets (`@id` and name), and within each record set, the available fields (`@id` and name). All lookups reference their Croissant `@id`.

In [ ]:
# List available record sets and their fields (by @id)

record_sets = dataset.metadata.record_sets

print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    print(f"  Name: {rs.get('name', '[No name]')}")
    # Fields in this record set
    if 'fields' in rs:
        print("  Fields:")
        for f in rs['fields']:
            print(f"    - @id: {f['@id']}, name: {f.get('name', '[No name]')}, dataType: {f.get('dataType', '[?]')}")
    print("")

## 3. Data Extraction
Let's load each record set into a pandas DataFrame, referencing by their `@id` fields. Replace `<record_set_id>` by the actual `@id` as listed above to select a table/record set.

We'll show all available columns/fields for each extracted record set for inspection.

In [ ]:
# Find all available record set @id's
record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]

# Load each record set into a DataFrame by @id
dataframes = {}
for recset_id in record_set_ids:
    try:
        recs = list(dataset.records(record_set=recset_id))
        df = pd.DataFrame(recs)
        dataframes[recset_id] = df
        print(f"Loaded {len(df)} records for record set {recset_id}")
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2),"\n")
    except Exception as e:
        print(f"Could not load record set {recset_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Pick one main record set and select numeric field(s) for analysis. We'll filter, normalize, and (optionally) group data. All references are made via Croissant schema `@id` fields. Please adjust the variables below to fit the desired field from your inspection above.

Below we select the first available record set and a numeric-looking field for our example.

In [ ]:
# Pick one record set (use its @id as found)
selected_record_set = record_set_ids[0]  # You can change this
df = dataframes[selected_record_set]

# Show columns for user reference (their @id or name by the Croissant spec)
print(f"Columns in {selected_record_set}: {df.columns.tolist()}")

# --- Example: Pick a numeric field (column) ---
# You can adjust this to any numeric column appropriate for your dataset
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if numeric_field is None:
    raise Exception('No numeric field found for EDA demonstration.')

print(f'Using numeric field: {numeric_field}')

# Filtering - keep only rows where the value is greater than a threshold
threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records in {selected_record_set} with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field (z-score normalization)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
    filtered_df[numeric_field].std()
)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optional: Pick a categorical field for grouping (if present)
group_field = None
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]) and col != numeric_field:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:\n")
    print(grouped_df.head())

## 5. Visualization
Below we show an example of simple visualizations, such as the distribution of a numeric field, or a scatterplot if more than one numeric field is present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f'Distribution of {numeric_field} in record set {selected_record_set}')
plt.xlabel(numeric_field)
plt.show()

# If more than one numeric field exists, show a pairplot or scatterplot
numeric_cols = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if len(numeric_cols) >= 2:
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
    plt.title(f'Scatterplot: {numeric_cols[0]} vs {numeric_cols[1]}')
    plt.xlabel(numeric_cols[0])
    plt.ylabel(numeric_cols[1])
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library, referencing all dataset objects by their Croissant schema `@id`. We loaded each record set, performed example data cleaning and normalization, and visualized key numeric attributes. For best analysis, always refer to the dataset documentation, field `@id`s, and schema specifications.